# M2.3 · SVM: hinge loss ve margin

Olasılık yerine margin. İkili ve çok sınıflı hinge kaybını elle hesaplıyor, saf Python'da alt-gradyanla lineer SVM eğitiyor, CS231n'in vektörize çok sınıflı SVM kaybını NumPy'da yazıyor, PyTorch'ta `multi_margin_loss` ile karşılaştırıyor ve en sonda RBF kernel matrisiyle doğrusal olmayan bir sınırı döngüsüz öğreniyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m2-3-svm.html](https://m-gonul.github.io/pytorch-lab/m2-3-svm.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m2_3_svm.py
CHECKS = {
 "b": "import random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def hand():\n        L, gw, gb = need(ns, \"hinge_grad\")([[2, 1], [0, 1], [1, 1]], [1, -1, 1], [1.0, -1.0], 0.0, 0.0)\n        close([L, gw[0], gw[1], gb], [1 / 3, -1 / 3, -1 / 3, -1 / 3])\n        L2, gw2, _ = need(ns, \"hinge_grad\")([[2, 1]], [1], [1.0, -1.0], 0.0, 0.5)\n        close([L2, gw2[0], gw2[1]], [1.0, 1.0, -1.0], what=\"yalnız ceza (λ = 0.5)\")\n    R.test(\"hinge_grad: elle örnek ve L2 cezası\", hand)\n    R.test(\"multiclass_hinge = 2.9\", lambda: close(need(ns, \"multiclass_hinge\")([3.2, 5.1, -1.7], 0), 2.9))\n\n    def fit():\n        rng = random.Random(0)\n        X, y = [], []\n        for _ in range(80):\n            c = rng.choice([-1, 1])\n            X.append([rng.gauss(1.5 * c, 0.8), rng.gauss(-1.0 * c, 0.8)]); y.append(c)\n        w, b = need(ns, \"fit_svm\")(X, y, lam=0.01, lr=0.1, steps=500)\n        acc = sum((sum(a * b_ for a, b_ in zip(w, x)) + b > 0) == (t > 0) for x, t in zip(X, y)) / 80\n        ensure(acc >= 0.9, f\"doğruluk {acc:.2f}\")\n        return f\"doğruluk {acc:.3f}, w = [{w[0]:.2f}, {w[1]:.2f}]\"\n    R.test(\"fit_svm iki kümeyi ayırıyor\", fit)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need\n\n\ndef naive(W, X, y, reg, delta=1.0):\n    n, C = X.shape[0], W.shape[1]\n    loss, dW = 0.0, np.zeros_like(W)\n    for i in range(n):\n        s = X[i] @ W\n        for j in range(C):\n            if j == y[i]:\n                continue\n            m = s[j] - s[y[i]] + delta\n            if m > 0:\n                loss += m; dW[:, j] += X[i]; dW[:, y[i]] -= X[i]\n    return loss / n + reg * (W * W).sum(), dW / n + 2 * reg * W\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    rng = np.random.default_rng(1)\n    W = rng.normal(size=(8, 4)) * 0.3\n    X = rng.normal(size=(30, 8))\n    y = rng.integers(0, 4, 30)\n\n    def eq():\n        L, dW = need(ns, \"svm_loss\")(W, X, y, reg=0.05)\n        Ln, dWn = naive(W, X, y, 0.05)\n        close(L, Ln, what=\"kayıp\"); close(dW, dWn, what=\"dW\")\n    R.test(\"naive (çift döngü) ile aynı\", eq)\n\n    def num():\n        h = 1e-6\n        f = lambda Wp: need(ns, \"svm_loss\")(Wp, X, y, 0.05)[0]\n        _, dW = need(ns, \"svm_loss\")(W, X, y, 0.05)\n        for (i, j) in [(0, 0), (3, 2), (7, 1)]:\n            E = np.zeros_like(W); E[i, j] = h\n            close(dW[i, j], (f(W + E) - f(W - E)) / (2 * h), rtol=1e-4, atol=1e-6, what=f\"dW[{i},{j}]\")\n    R.test(\"sayısal gradyanla aynı\", num)\n    R.test(\"elle örnek: 2.9\", lambda: close(need(ns, \"svm_loss\")(np.eye(3), np.array([[3.2, 5.1, -1.7]]), np.array([0]))[0], 2.9))\n",
 "d": "import ast\nimport torch\nimport torch.nn.functional as F\nfrom torch import nn\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n\n    def nb():\n        fn = next((n for n in ast.walk(ast.parse(src)) if isinstance(n, ast.FunctionDef) and n.name == \"svm_loss_torch\"), None)\n        ensure(fn is not None, \"`svm_loss_torch` yok\")\n        ensure(\"multi_margin_loss\" not in ast.unparse(fn), \"svm_loss_torch'u elle yaz\")\n    R.test(\"hazır multi_margin_loss kullanılmıyor\", nb)\n    g = torch.Generator().manual_seed(0)\n    S = torch.randn(25, 6, generator=g, dtype=torch.float64) * 2\n    y = torch.randint(0, 6, (25,), generator=g)\n    R.test(\"= C · F.multi_margin_loss\", lambda: close(need(ns, \"svm_loss_torch\")(S, y), 6 * F.multi_margin_loss(S, y)))\n    R.test(\"elle örnek 2.9\", lambda: close(need(ns, \"svm_loss_torch\")(torch.tensor([[3.2, 5.1, -1.7]]), torch.tensor([0])), 2.9))\n\n    def tr():\n        centers = torch.tensor([[0.0, 3.0], [-3.0, -2.0], [3.0, -2.0]])\n        yy = torch.arange(3).repeat_interleave(60)\n        X = centers[yy] + torch.randn(180, 2, generator=g)\n        torch.manual_seed(0)\n        m = need(ns, \"train_linear_svm\")(X, yy, 3)\n        ensure(isinstance(m, nn.Linear), \"nn.Linear bekleniyordu\")\n        acc = (m(X).argmax(1) == yy).float().mean().item()\n        ensure(acc >= 0.9, f\"doğruluk {acc:.2f}\")\n        return f\"doğruluk {acc:.3f}\"\n    R.test(\"train_linear_svm\", tr)\n",
 "e": "import torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef circles(g, n):\n    r = torch.cat([torch.full((n,), 1.0), torch.full((n,), 3.0)]) + 0.2 * torch.randn(2 * n, generator=g)\n    t = torch.rand(2 * n, generator=g) * 6.283185\n    return torch.stack([r * t.cos(), r * t.sin()], 1), torch.cat([torch.ones(n), -torch.ones(n)])\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"rbf_kernel/predict_kernel: döngü yok\", lambda: (max_loops(src, 0, func=\"rbf_kernel\"), max_loops(src, 0, func=\"predict_kernel\")))\n    R.test(\"fit_kernel_svm: yalnız adım döngüsü\", lambda: max_loops(src, 1, func=\"fit_kernel_svm\"))\n    g = torch.Generator().manual_seed(0)\n    A, B = torch.randn(13, 3, generator=g, dtype=torch.float64), torch.randn(7, 3, generator=g, dtype=torch.float64)\n    R.test(\"rbf_kernel doğru\", lambda: close(need(ns, \"rbf_kernel\")(A, B, 0.7), torch.exp(-0.7 * torch.cdist(A, B) ** 2), rtol=1e-9, atol=1e-12))\n\n    def circ():\n        X, y = circles(g, 120)\n        Xt, yt = circles(g, 100)\n        alpha, b = need(ns, \"fit_kernel_svm\")(X, y, gamma=0.5)\n        acc = (need(ns, \"predict_kernel\")(X, alpha, b, 0.5, Xt) == yt).float().mean().item()\n        ensure(acc >= 0.95, f\"test doğruluğu {acc:.2f}\")\n        w = torch.linalg.lstsq(torch.cat([X, torch.ones(240, 1)], 1), y[:, None]).solution\n        lin = ((torch.cat([Xt, torch.ones(200, 1)], 1) @ w).sign().view(-1) == yt).float().mean().item()\n        nsv = int((alpha.abs() > 1e-6).sum())\n        return f\"iç içe çemberler: kernel SVM test doğruluğu {acc:.3f}, lineer model {lin:.3f}\"\n    R.test(\"kernel SVM doğrusal olmayan sınırı öğreniyor\", circ)\n    if \"rbf_kernel\" not in ns:\n        return\n    Xb = torch.randn(1500, 10, generator=g)\n    t_loop = bench(lambda: torch.stack([torch.exp(-0.5 * ((Xb - x) ** 2).sum(1)) for x in Xb]), repeat=3, number=1)\n    t_vec = bench(lambda: ns[\"rbf_kernel\"](Xb, Xb, 0.5), repeat=3)\n    R.table(\"1500 × 1500 RBF Gram matrisi\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for satır: exp(−γ‖x − X‖²)\", fmt_time(t_loop), \"1×\"], [\"senin rbf_kernel\", fmt_time(t_vec), f\"{t_loop / t_vec:,.0f}× hızlı\"]],\n            note=\"Kernel yöntemlerinin darboğazı n × n matris: n = 10⁵ için 40 GB (float32). Büyük veride yaklaşık yöntemler (Nyström, random features) kullanılır.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m2_3_svm as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Logistic regression her örneğin olasılığını iyileştirmeye çalışır; **SVM** ise yalnız sınıflar arasında güvenli bir boşluk (margin) ister. $y\in\{-1,+1\}$ ve $f(x)=w^\top x+b$ için hinge kaybı:

$$
\ell(x,y)=\max\big(0,\;1-y\,f(x)\big),\qquad
L=\frac1n\sum_i\ell_i+\lambda\lVert w\rVert^2 .
$$

$yf(x)\ge1$ olan örnekler kayba katkı yapmaz. Kayıp $yf(x)=1$'de kırılır; orada türev yok ama bir alt-gradyan var: aktif örnekler için $\partial\ell/\partial w=-y\,x$, diğerleri için 0.

Çok sınıflı sürüm (Weston–Watkins, CS231n): $s=Wx$ skorlar,

$$
\ell=\sum_{j\ne y}\max(0,\;s_j-s_y+\Delta).
$$

Doğrusal olmayan sınırlar için kernel trick: $f(x)=\sum_i\alpha_i\,k(x_i,x)+b$. RBF kernel $k(x,x')=e^{-\gamma\lVert x-x'\rVert^2}$ ile model, her eğitim noktasına bir "tümsek" koyar.

:::türetim Marj neden $2/\|w\|$?
Karar sınırı $w^\top x+b=0$. Bir $x_0$ noktasının sınıra dik uzaklığı $\frac{|w^\top x_0+b|}{\|w\|}$: $x_0$'dan $w$ yönünde (sınırın normali) $t$ kadar gidip sınıra ulaşmak için $w^\top(x_0-t\frac{w}{\|w\|})+b=0\Rightarrow t=\frac{w^\top x_0+b}{\|w\|}$.

Ölçek serbestliği: $(w,b)$'yi $(cw,cb)$ yapmak sınırı değiştirmez. Bu serbestliği en yakın noktalarda $y_i(w^\top x_i+b)=1$ olacak şekilde sabitleriz (kanonik biçim). O zaman en yakın noktanın uzaklığı $1/\|w\|$, iki sınıf arasındaki şerit $2/\|w\|$.

Marjı büyütmek = $\|w\|$'yi küçültmek: $\min\frac12\|w\|^2$, kısıt $y_i(w^\top x_i+b)\ge1$ (sert marj). Ayrılamayan veride gevşek değişkenler $\xi_i\ge0$ eklenir ve ceza $C\sum\xi_i$; optimumda $\xi_i=\max(0,1-y_if(x_i))$ — **hinge kaybı**. Böylece yumuşak marjlı SVM $\frac12\|w\|^2+C\sum_i\max(0,1-y_if(x_i))$ olur: L2 düzenlileştirilmiş hinge regresyonu ($\lambda=1/C$). Destek vektörleri, kısıtı aktif olan (marjda ya da içinde) noktalardır; diğerlerini silmek çözümü değiştirmez.
:::

## 1 · Elle

<!-- çözüm -->

İkili örnekte üç noktanın margin'ini hesapla. $x=(2,1)$ tam margin sınırında (kayıp 0, gradyan 0): bu bir **destek vektörü**. $(1,1)$ karar sınırının üzerinde ($f=0$): kayıp 1 ve gradyan $w$'yi bu noktayı pozitif tarafa itecek yönde.

Çok sınıflı örnek CS231n'den: kedi resminin skorları [kedi 3.2, araba 5.1, kurbağa −1.7]. Araba, kediyi $\Delta=1$'lik güvenlik payıyla geçmediği için ceza alıyor; kurbağa zaten çok geride. Doğru sınıfın skoru, ihlal eden her sınıf için bir kez aşağı itilir (gradyan $-1$ katı).


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İkili: $w=[1,-1]$, $b=0$; $f(x)=w\cdot x+b$, kayıp $\max(0,1-yf(x))$**

- `m1`: $x=(2,1)$, $y=+1$: margin $yf(x)$
- `h1`: … hinge
- `m3`: $x=(1,1)$, $y=+1$: margin
- `h3`: … hinge
- `gw1`: … $\partial\ell/\partial w_1$

**Çok sınıflı (CS231n): skorlar $s=[3.2,\,5.1,\,-1.7]$, doğru sınıf $y=0$, $\Delta=1$**

- `l1`: $\max(0,s_1-s_0+\Delta)$
- `l2`: $\max(0,s_2-s_0+\Delta)$
- `L`: toplam kayıp
- `nviol`: doğru sınıfın skor gradyanı $\partial L/\partial s_0$

In [ ]:
cevaplar = dict(
    m1=None,
    h1=None,
    m3=None,
    h3=None,
    gw1=None,
    l1=None,
    l2=None,
    L=None,
    nviol=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"m1":[1,0],"h1":[0,0],"m3":[0,0],"h3":[1,0],"gw1":[-1,0],"l1":[2.9,0.001],"l2":[0,0.001],"L":[2.9,0.001],"nviol":[-1,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

Saf Python, ikili lineer SVM. `y` elemanları $\pm1$.

- `hinge_grad(X, y, w, b, lam)` → `(L, gw, gb)`; $L=\frac1n\sum\max(0,1-y_if_i)+\lambda\lVert w\rVert^2$. Alt-gradyan: aktif ($y_if_i<1$) örnekler $-y_ix_i/n$ katkı yapar, ceza $2\lambda w$.
- `fit_svm(X, y, lam=0.01, lr=0.1, steps=500)` → `(w, b)`; tam batch alt-gradyan inişi.
- `multiclass_hinge(scores, y, delta=1.0)` — tek örnek, CS231n kaybı.

In [ ]:
%%writefile stage_b.py
def hinge_grad(X, y, w, b, lam):
    ...


def fit_svm(X, y, lam=0.01, lr=0.1, steps=500):
    ...


def multiclass_hinge(scores, y, delta=1.0):
    ...


print(hinge_grad([[2, 1], [0, 1], [1, 1]], [1, -1, 1], [1.0, -1.0], 0.0, 0.0))
print(multiclass_hinge([3.2, 5.1, -1.7], 0))   # 2.9

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

CS231n `svm_loss_vectorized`: $W\in\mathbb R^{d\times C}$, $X\in\mathbb R^{n\times d}$, $S=XW$.

$$
M=\max\big(0,\;S-s_y\mathbf 1^\top+\Delta\big),\quad M_{i,y_i}=0,\qquad L=\frac1n\sum M+\text{reg}\lVert W\rVert^2 .
$$

Gradyan için ihlal göstergesi $B=[M>0]$; doğru sınıf sütununa satırdaki ihlal sayısının negatifi yazılır, $dW=X^\top B/n+2\,\text{reg}\,W$.

- `svm_loss(W, X, y, reg=0.0, delta=1.0)` → `(loss, dW)`. **Döngü yok.** Kontrol, naive (çift döngülü) referansla ve sayısal gradyanla karşılaştırır.

In [ ]:
%%writefile stage_c.py
import numpy as np


def svm_loss(W, X, y, reg=0.0, delta=1.0):
    """→ (loss, dW). Döngü yok."""
    ...


rng = np.random.default_rng(0)
W = rng.normal(size=(5, 3)) * 0.01
X = rng.normal(size=(10, 5))
y = rng.integers(0, 3, 10)
print(svm_loss(W, X, y, reg=0.1))

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

- `svm_loss_torch(S, y, delta=1.0)` — skorlardan ortalama CS231n kaybı, **elle** (`F.multi_margin_loss` kullanma). Kontrol, sonucun `C * F.multi_margin_loss(S, y)` ile aynı olduğunu doğrular.
- `train_linear_svm(X, y, C, steps=500, lr=0.1, wd=1e-3)` → `nn.Linear(d, C)`; kayıp senin `svm_loss_torch`'un, ceza `weight_decay` ile.

In [ ]:
%%writefile stage_d.py
import torch
import torch.nn.functional as F
from torch import nn


def svm_loss_torch(S, y, delta=1.0):
    """S: (n, C) skorlar → ortalama çok sınıflı hinge (CS231n). Elle."""
    ...


def train_linear_svm(X, y, C, steps=500, lr=0.1, wd=1e-3):
    ...


S = torch.tensor([[3.2, 5.1, -1.7]])
print(svm_loss_torch(S, torch.tensor([0])))   # 2.9

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Kernel SVM'i primal'de, $\alpha$ üzerinden eğit: $K_{ij}=k(x_i,x_j)$, $f=K\alpha+b$,

$$
L(\alpha,b)=\frac1n\sum_i\max(0,1-y_if_i)+\lambda\,\alpha^\top K\alpha .
$$

- `rbf_kernel(A, B, gamma)` → `(n_A, n_B)`; mesafe açılımıyla (M2.1), **döngü yok**.
- `fit_kernel_svm(X, y, gamma, lam=1e-3, steps=300, lr=0.5)` → `(alpha, b)`; tam batch alt-gradyan, yalnız adım döngüsü. $\nabla_\alpha=\frac1nK^\top(-y\odot a)+2\lambda K\alpha$, $a$ aktiflik maskesi.
- `predict_kernel(Xtr, alpha, b, gamma, X)` → işaretler $\pm1$.

Kontrol: iç içe iki çember (lineer olarak ayrılamaz) üzerinde kernel SVM ≥ 0.95 doğruluk alırken lineer SVM şans seviyesinde kalır.

In [ ]:
%%writefile stage_e.py
import torch


def rbf_kernel(A, B, gamma):
    ...


def fit_kernel_svm(X, y, gamma, lam=1e-3, steps=300, lr=0.5):
    """y ∈ {−1, +1}. → (alpha: (n,), b). Yalnız adım döngüsü."""
    ...


def predict_kernel(Xtr, alpha, b, gamma, X):
    ...


g = torch.Generator().manual_seed(0)
r = torch.cat([torch.full((100,), 1.0), torch.full((100,), 3.0)]) + 0.2 * torch.randn(200, generator=g)
t = torch.rand(200, generator=g) * 6.2832
X = torch.stack([r * t.cos(), r * t.sin()], 1)
y = torch.cat([torch.ones(100), -torch.ones(100)])
alpha, b = fit_kernel_svm(X, y, gamma=0.5)
print((predict_kernel(X, alpha, b, 0.5, X) == y).float().mean())

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Platt (1998): Sequential Minimal Optimization

[Platt — Sequential Minimal Optimization: A Fast Algorithm for Training Support Vector Machines (Microsoft Research TR 1998)](https://www.microsoft.com/en-us/research/publication/sequential-minimal-optimization-a-fast-algorithm-for-training-support-vector-machines/)

SMO'yu sıfırdan yaz: iki Lagrange çarpanını analitik olarak güncelleme (kırpma sınırları L, H), ikinci çarpan seçme sezgisi (|E₁ − E₂| maksimum), bias güncellemesi ve KKT ihlali kontrolü. RBF çekirdekli ikili bir problemde sonucu sklearn.svm.SVC (libsvm) ile karşılaştır: aynı destek vektörleri ve karar fonksiyonu; n = 10³…10⁴ için süre ölçeklemesi.


**Adımlar**

1. Hata önbelleği (error cache) ekleyip etkisini ölç.

**Kabul ölçütleri**

- Karar fonksiyonu libsvm ile 10⁻³ içinde aynı.
- KKT koşulları tolerans içinde sağlanıyor.

## Kaynaklar

- [CS231n — Linear classification: Multiclass SVM loss](https://cs231n.github.io/linear-classify/#svm) — MIT
- [CS231n Assignment 1 — svm_loss_naive / svm_loss_vectorized](https://cs231n.github.io/assignments2025/assignment1/) — MIT
- [Stanford CS229 — Lecture notes: Support Vector Machines, kernels](https://cs229.stanford.edu/main_notes.pdf) — doğrulanmadı
- [Shalev-Shwartz et al. — Pegasos: Primal Estimated sub-GrAdient SOlver for SVM](https://home.ttic.edu/~nati/Publications/PegasosMPB.pdf) — ©